#WEB

In [1]:
!git clone https://github.com/duyvo26/python_hoc_may_end_mon

Cloning into 'python_hoc_may_end_mon'...
remote: Enumerating objects: 174, done.
remote: Counting objects: 100% (174/174), done.
remote: Compressing objects: 100% (111/111), done.
remote: Total 174 (delta 109), reused 124 (delta 59), pack-reused 0 (from 0)
Receiving objects: 100% (174/174), 92.91 KiB | 2.27 MiB/s, done.
Resolving deltas: 100% (109/109), done.


In [2]:
%cd /kaggle/working/python_hoc_may_end_mon

!pip install -r requirements.txt

/kaggle/working/python_hoc_may_end_mon
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.5/117.5 kB 7.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.8/108.8 kB 6.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.4/114.4 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/135.7 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.5/110.5 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.7/323.7 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.7/117.7 kB 7.4 MB/s eta 0:00:00


In [3]:
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

--2026-04-27 18:44:14--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
Resolving github.com (github.com)... 140.82.121.4
Connecting to github.com (github.com)|140.82.121.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2026.3.0/cloudflared-linux-amd64 [following]
--2026-04-27 18:44:15--  https://github.com/cloudflare/cloudflared/releases/download/2026.3.0/cloudflared-linux-amd64
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/106867604/731ab2f8-6b77-4adb-a7b3-1104525e9d72?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-04-27T19%3A31%3A10Z&rscd=attachment%3B+filename%3Dcloudflared-linux-amd64&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-04-27T1

In [4]:
import re
import subprocess
import threading
import time
import sys
import os
import signal
import urllib.request

PORT = 5000
URL_PATTERN = re.compile(r"(https://[^\s]+\.trycloudflare\.com)")

# =======================
# CONFIG
# =======================
SHOW_WEB_LOG = "--log" in sys.argv  # 👈 mặc định tắt


public_url = None


# =======================
# Kill process on port
# =======================
def kill_port(port):
    try:
        result = subprocess.check_output(
            f"lsof -ti:{port}", shell=True, text=True
        ).strip()

        if result:
            pids = result.split("\n")
            for pid in pids:
                print(f"🔥 Killing PID {pid} on port {port}")
                os.kill(int(pid), signal.SIGKILL)
        else:
            print(f"✅ Port {port} is free")

    except subprocess.CalledProcessError:
        print(f"✅ Port {port} is free")


# =======================
# Check HTTP ready
# =======================
def is_port_ready(port):
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{port}", timeout=1) as res:
            return res.status in (200, 404)
    except:
        return False


# =======================
# Read Flask output
# =======================
def read_flask_output(proc):
    for line in proc.stdout:
        if SHOW_WEB_LOG:
            print("[FLASK]", line.strip())


# =======================
# Read Cloudflare output
# =======================
def read_cloudflare_output(proc):
    global public_url

    for line in proc.stdout:
        # print("[CLOUDFLARE]", line.strip())

        match = URL_PATTERN.search(line)
        if match:
            public_url = match.group(1)


# =======================
# MAIN
# =======================
def main():
    global public_url

    print("🚀 Starting setup...")

    # Kill port trước
    kill_port(PORT)
    time.sleep(1)

    # =======================
    # ENV Flask
    # =======================
    flask_env = os.environ.copy()

    if not SHOW_WEB_LOG:
        flask_env["FLASK_ENV"] = "production"

    # =======================
    # Start Flask
    # =======================
    flask_process = subprocess.Popen(
        ["python3", "server.py"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=flask_env,
    )

    # =======================
    # Start Cloudflare
    # =======================
    cloudflare_process = subprocess.Popen(
        [
            "./cloudflared",
            "tunnel",
            "--url",
            f"http://localhost:{PORT}",
            "--no-autoupdate",
            "--loglevel",
            "info",
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    # =======================
    # Threads đọc log
    # =======================
    threading.Thread(
        target=read_flask_output, args=(flask_process,), daemon=True
    ).start()

    threading.Thread(
        target=read_cloudflare_output, args=(cloudflare_process,), daemon=True
    ).start()

    # =======================
    # Wait until ready
    # =======================
    timeout = 60
    start_time = time.time()

    server_ready = False

    print("⏳ Waiting for server + tunnel...")

    while True:
        if flask_process.poll() is not None:
            print("❌ Flask process stopped!")
            sys.exit(1)

        if cloudflare_process.poll() is not None:
            print("❌ Cloudflared process stopped!")
            sys.exit(1)

        # Check HTTP thật
        if not server_ready and is_port_ready(PORT):
            print("✅ HTTP server is ready")
            server_ready = True

        # Khi cả 2 ok
        if server_ready and public_url:
            print("\n🎉 ALL READY")
            time.sleep(15)
            print(f"🌍 Public URL: {public_url}\n")
            break

        if time.time() - start_time > timeout:
            print("⏰ Timeout: Could not get public URL")
            break

        time.sleep(0.5)


# =======================
# Run
# =======================
if __name__ == "__main__":
    main()

🚀 Starting setup...
✅ Port 5000 is free
⏳ Waiting for server + tunnel...
[CLOUDFLARE] 2026-04-27T18:44:18Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
[CLOUDFLARE] 2026-04-27T18:44:18Z INF Requesting new quick Tunnel on trycloudflare.com...
[CLOUDFLARE] 2026-04-27T18:44:21Z INF +--------------------------------------------------------------------------------------------+
[CLOUDFLARE] 2026-04-27T18:44:21Z INF |  Your quick Tunnel has been created! Visit it at 